# Grand Minima Feature Analysis: Triple Cross-Validation
# Grand Minima 특징 분석: 3중 교차 검증

**Paper / 논문**: Lee, E. H. & Lee, D. Y. (2007)
*Advances in Space Research*, 40, 942–950.

## 목표 / Objectives

1. **그림 1 재현** — 흑점 + 오로라 + Δ¹⁴C의 3중 비교 시각화
2. **Grand Minima 지속 기간 패턴** — Oort → Wolf → Spörer → Maunder → Dalton
3. **Grand Minima 간격 분석** — super-secular cycle 시각화

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams["figure.figsize"] = (14, 6)
rcParams["font.size"] = 11
rcParams["axes.grid"] = True
rcParams["grid.alpha"] = 0.3

## 1. 데이터 준비 / Data Preparation

그림 1의 3개 패널을 위한 데이터를 준비합니다:
- (a) 흑점: 표 1에서 디지타이징한 연간 흑점 기록 수
- (b) 오로라: 표 2에서 디지타이징한 연간 오로라 기록 수 (한국+중국)
- (c) Δ¹⁴C: Stuiver et al. (1998) IntCal98 데이터의 근사값

In [ ]:
# Sunspot records per decade (digitized from Figure 1a, log scale)
# Combined Korea + China records
sunspot_decades = {
    1070: 1, 1080: 1, 1100: 3, 1110: 2, 1120: 3, 1130: 5, 1140: 3,
    1150: 3, 1160: 2, 1170: 3, 1180: 6, 1190: 1, 1200: 5, 1210: 1,
    1250: 2, 1270: 1, 1280: 1, 1350: 3, 1360: 8, 1370: 15, 1380: 5,
    1400: 1, 1500: 1, 1510: 1, 1560: 2, 1590: 1, 1600: 5, 1610: 8,
    1620: 10, 1630: 8, 1640: 5, 1650: 3, 1660: 3, 1700: 1, 1720: 5,
    1730: 2, 1740: 2,
}

# Aurora records per decade (digitized from Figure 1b)
aurora_decades = {
    1000: 1, 1010: 3, 1020: 2, 1050: 1, 1070: 4, 1080: 2, 1090: 3,
    1100: 10, 1110: 5, 1120: 12, 1130: 8, 1140: 3, 1150: 1, 1170: 15,
    1180: 13, 1190: 5, 1200: 4, 1210: 5, 1220: 9, 1230: 3, 1240: 1,
    1250: 13, 1260: 10, 1270: 5, 1280: 4, 1290: 3, 1300: 2, 1310: 2,
    1320: 5, 1350: 5, 1360: 22, 1370: 18, 1380: 5, 1390: 5, 1400: 6,
    1460: 3, 1470: 1, 1490: 1, 1500: 2, 1510: 8, 1520: 15, 1530: 30,
    1540: 22, 1550: 28, 1560: 18, 1570: 3, 1580: 2, 1590: 5, 1600: 12,
    1610: 8, 1620: 30, 1630: 5, 1640: 3, 1650: 3, 1660: 8, 1670: 3,
    1680: 5, 1690: 2, 1700: 5, 1710: 5, 1720: 10, 1730: 2, 1740: 2,
    1750: 3, 1760: 1, 1770: 1,
}

# Delta-14C data (approximate, digitized from Figure 1c)
# Based on Stuiver et al. (1998) IntCal98, 10-year averages
delta14c_years = np.arange(1000, 2010, 10)
delta14c_values = np.array([
    -15, -12, -8, -5, -2, 0, 2, 0, -3, -5,     # 1000-1090
    -8, -10, -8, -5, -3, 0, 3, 5, 3, 0,         # 1100-1190
    -3, -5, -3, 0, 2, 5, 8, 10, 8, 5,           # 1200-1290
    3, 0, -2, -3, -2, 0, 2, 5, 8, 10,           # 1300-1390
    12, 15, 18, 20, 18, 15, 12, 10, 8, 5,       # 1400-1490 (Spörer peak)
    3, 0, -5, -8, -10, -12, -10, -8, -5, -3,    # 1500-1590
    0, 3, 5, 8, 12, 15, 12, 8, 5, 3,            # 1600-1690 (Maunder peak)
    0, -3, -5, -8, -5, -3, 0, -3, -5, -8,       # 1700-1790
    -10, -12, -15, -18, -20, -22, -23, -23, -22, -20, # 1800-1890
    -18, -15, -12, -10, -8, -5, -3, 0, 2, 5,    # 1900-1990
    8                                              # 2000
])

# Grand Minima definitions
grand_minima = {
    "Oort":    {"start": 1010, "end": 1050, "color": "blue"},
    "Wolf":    {"start": 1280, "end": 1340, "color": "green"},
    "Spörer":  {"start": 1410, "end": 1510, "color": "orange"},
    "Maunder": {"start": 1645, "end": 1715, "color": "red"},
}

print("Data prepared for 3-panel figure.")

## 2. 그림 1 재현: 3중 비교 / Reproduce Figure 1: Triple Comparison

논문의 핵심 그림. 3개 패널: (a) 흑점, (b) 오로라, (c) Δ¹⁴C.
Grand Minima가 세 독립적 데이터에서 동시에 나타나는 것을 확인합니다.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Helper: shade Grand Minima on all panels
def shade_minima(ax, label_y_frac=0.85):
    for name, info in grand_minima.items():
        ax.axvspan(info["start"], info["end"],
                   alpha=0.15, color=info["color"])
        mid = (info["start"] + info["end"]) / 2
        ylim = ax.get_ylim()
        y_pos = ylim[0] + (ylim[1] - ylim[0]) * label_y_frac
        ax.text(mid, y_pos, name, ha="center", fontsize=9,
                color=info["color"], fontweight="bold")

# --- Panel (a): Sunspot records (log scale) ---
ss_years = sorted(sunspot_decades.keys())
ss_counts = [sunspot_decades[y] for y in ss_years]

axes[0].bar(ss_years, ss_counts, width=9, color="black", alpha=0.8)
axes[0].set_yscale("log")
axes[0].set_ylabel("Number")
axes[0].set_ylim(0.5, 200)
axes[0].set_title("(a) Sunspot Records — Korea + China\n"
                   "흑점 기록 — 한국 + 중국")
shade_minima(axes[0])
axes[0].text(1050, 100, "Sunspot", fontsize=14, fontweight="bold")

# --- Panel (b): Aurora records ---
au_years = sorted(aurora_decades.keys())
au_counts = [aurora_decades[y] for y in au_years]

axes[1].bar(au_years, au_counts, width=9, color="black", alpha=0.8)
axes[1].set_ylabel("Number")
axes[1].set_title("(b) Aurora Records — Korea + China\n"
                   "오로라 기록 — 한국 + 중국")
shade_minima(axes[1])
axes[1].text(1050, 25, "Aurora", fontsize=14, fontweight="bold")

# --- Panel (c): Delta-14C ---
axes[2].plot(delta14c_years, delta14c_values, "k-", linewidth=1.5)
axes[2].fill_between(delta14c_years, 0, delta14c_values,
                      where=(delta14c_values > 0),
                      alpha=0.2, color="red", label="Δ¹⁴C > 0 (weak Sun)")
axes[2].fill_between(delta14c_years, 0, delta14c_values,
                      where=(delta14c_values <= 0),
                      alpha=0.2, color="blue", label="Δ¹⁴C < 0 (strong Sun)")
axes[2].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
axes[2].set_ylabel("Δ¹⁴C (‰)")
axes[2].set_xlabel("Year")
axes[2].set_title("(c) Δ¹⁴C Variation (Stuiver et al. 1998)\n"
                   "Δ¹⁴C 변동 — 양수=태양활동 약함, 음수=태양활동 강함")
axes[2].legend(loc="lower left", fontsize=9)
shade_minima(axes[2], label_y_frac=0.9)

axes[2].set_xlim(980, 2020)

fig.suptitle("Figure 1: Triple Cross-Validation of Grand Minima\n"
             "그림 1: Grand Minima의 3중 교차 검증",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Grand Minima 지속 기간 패턴 / Duration Pattern

논문의 핵심 발견: Grand Minima 지속 기간이 Spörer를 정점으로 증가→감소하는 비대칭 패턴.
Key finding: Duration follows an asymmetric increase→decrease pattern peaking at Spörer.

In [ ]:
# Grand Minima data
minima_data = {
    "Oort":    {"start": 1010, "end": 1050, "mid": 1030, "duration": 40},
    "Wolf":    {"start": 1280, "end": 1340, "mid": 1310, "duration": 60},
    "Spörer":  {"start": 1410, "end": 1510, "mid": 1460, "duration": 100},
    "Maunder": {"start": 1645, "end": 1715, "mid": 1680, "duration": 70},
    "Dalton":  {"start": 1790, "end": 1830, "mid": 1810, "duration": 40},
}

names = list(minima_data.keys())
mids = [minima_data[n]["mid"] for n in names]
durations = [minima_data[n]["duration"] for n in names]
starts = [minima_data[n]["start"] for n in names]
ends = [minima_data[n]["end"] for n in names]

# Compute intervals between successive minima (mid-to-mid)
intervals = [mids[i+1] - mids[i] for i in range(len(mids)-1)]
interval_labels = [f"{names[i]}→{names[i+1]}" for i in range(len(names)-1)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- (a) Duration pattern ---
colors = ["blue", "green", "orange", "red", "purple"]
axes[0].bar(names, durations, color=colors, alpha=0.7, edgecolor="black")
axes[0].plot(names, durations, "ko--", markersize=8, linewidth=2)

for i, (name, dur) in enumerate(zip(names, durations)):
    axes[0].annotate(f"~{dur} yr", xy=(i, dur + 3),
                     ha="center", fontsize=11, fontweight="bold")

axes[0].set_ylabel("Duration (years)")
axes[0].set_title("(a) Grand Minima Duration Pattern\n"
                   "Grand Minima 지속 기간 패턴")
axes[0].set_ylim(0, 120)

# Add arrow showing increase→decrease pattern
axes[0].annotate("", xy=(2, 110), xytext=(0, 50),
                 arrowprops=dict(arrowstyle="->", color="darkred", lw=2))
axes[0].annotate("", xy=(4, 50), xytext=(2, 110),
                 arrowprops=dict(arrowstyle="->", color="darkblue", lw=2))
axes[0].text(1, 95, "increase\n증가", ha="center", color="darkred", fontsize=10)
axes[0].text(3, 95, "decrease\n감소", ha="center", color="darkblue", fontsize=10)

# --- (b) Intervals between successive minima ---
interval_mids = [(mids[i] + mids[i+1]) / 2 for i in range(len(mids)-1)]
axes[1].bar(interval_labels, intervals, color="gray", alpha=0.7, edgecolor="black")

for i, (label, intv) in enumerate(zip(interval_labels, intervals)):
    axes[1].annotate(f"~{intv} yr", xy=(i, intv + 5),
                     ha="center", fontsize=11, fontweight="bold")

axes[1].set_ylabel("Interval (years)")
axes[1].set_title("(b) Intervals Between Grand Minima\n"
                   "Grand Minima 간격")
axes[1].set_ylim(0, 300)
axes[1].tick_params(axis='x', rotation=15)

fig.suptitle("Grand Minima: Duration and Interval Analysis\n"
             "Grand Minima: 지속 기간 및 간격 분석",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print summary
print("="*65)
print("Grand Minima Summary / Grand Minima 요약")
print("="*65)
print(f"{'Name':10s} {'Start':>6s} {'End':>6s} {'Duration':>10s} {'Interval':>10s}")
print("-"*65)
for i, name in enumerate(names):
    d = minima_data[name]
    intv = f"→ {intervals[i]} yr" if i < len(intervals) else "—"
    print(f"{name:10s} {d['start']:>6d} {d['end']:>6d} {d['duration']:>8d} yr {intv:>10s}")
print("="*65)
print(f"\nSpörer Minimum is the LONGEST ({durations[2]} yr) — peak of the pattern.")
print(f"Pattern: {' → '.join(str(d) for d in durations)} yr")

## 4. 결과 요약 / Summary

### 구현 내용 / Implementation Summary

| 구현 항목 / Item | 논문 Figure | 결과 / Result |
|---|---|---|
| 3중 비교 시각화 | Fig. 1 (a,b,c) | 흑점·오로라·Δ¹⁴C에서 Grand Minima 일치 확인 |
| 지속 기간 패턴 | Section 4 | 40→60→100→70→40 yr (Spörer 정점) |
| 간격 분석 | Section 4 | 280→150→220→130 yr (불규칙) |

### 핵심 인사이트 / Key Insights

1. **3중 검증의 시각적 증거**: 세 완전히 독립적인 데이터 소스가 Grand Minima 시기에서 정확히 일치함이 한 그림으로 확인됨.
   Three independent data sources visually confirmed to align at Grand Minima timing.

2. **Spörer의 특별함**: 가장 긴 지속 기간(~100yr), 가장 큰 Δ¹⁴C 증가, 가장 희박한 역사 기록 — 3중으로 가장 극심한 Grand Minimum.
   Longest duration, largest Δ¹⁴C increase, sparsest records — triple evidence of most severe Grand Minimum.

3. **Super-secular cycle**: 지속 기간의 증가→감소 패턴은 수백 년~천 년 규모의 태양활동 초장주기를 시사.
   Duration pattern suggests a multi-century to millennial super-secular cycle in solar activity.